# 04. Calibración Probabilística, Aprendizaje Incremental (*Streaming*) y Benchmark Integral

**Módulo:** `05_Naive_Bayes`  
**Objetivo:** Analizar la patología de sobreconfianza probabilística en Naive Bayes y corregirla mediante `CalibratedClassifierCV` (Escalado de Platt y Regresión Isotónica) evaluando Curvas de Confiabilidad y el *Brier Score*; implementar arquitecturas de aprendizaje fuera de núcleo (*Out-of-Core / Streaming*) con `partial_fit` y `HashingVectorizer` para volúmenes masivos de datos; y ejecutar un benchmark multidimensional (calidad predictiva, escalabilidad temporal $\mathcal{O}(N \cdot D)$ y latencia de inferencia) frente a Regresión Logística, KNN, Árboles CART y Random Forest.

---

## 1. El Dilema de la Calibración en Naive Bayes: ¿Por Qué sus Probabilidades son Extremistas?

En los cuadernos anteriores demostramos que Naive Bayes puede alcanzar una alta exactitud de clasificación (buen *ranking* relativo) incluso cuando las características están correlacionadas. Sin embargo, cuando inspeccionamos las probabilidades devueltas por `.predict_proba(X)`, observamos un fenómeno alarmante: **el histograma de probabilidades se concentra masivamente cerca de `0.0000` y `1.0000`**.

### Demostración Algebraica de la Sobreconfianza
Consideremos un problema binario $Y \in \{0, 1\}$. La razón de probabilidades posteriores (*Posterior Odds*) en espacio logarítmico (el **Logit** estimado por Naive Bayes) es:

$$\ln \frac{\hat{P}(Y=1 \mid \mathbf{x})}{\hat{P}(Y=0 \mid \mathbf{x})} = \ln \frac{P(Y=1)}{P(Y=0)} + \sum_{j=1}^{D} \ln \frac{P(x_j \mid Y=1)}{P(x_j \mid Y=0)}$$

Supongamos el caso extremo donde una característica informativa $X_1$ está duplicada $m$ veces en el dataset (o existen $m$ características altamente colineales que aportan exactamente la misma señal de evidencia $\Delta = \ln \frac{P(x_1 \mid Y=1)}{P(x_1 \mid Y=0)} > 0$):
1. **Realidad condicional:** Dado que las $m$ variables contienen la misma información redundante, la contribución real al *log-odds* debería ser simplemente $\Delta$.
2. **Cálculo de Naive Bayes:** Al asumir independencia condicional, Naive Bayes suma $m$ veces la misma evidencia: $m \cdot \Delta$.
3. **Efecto en la Sigmoide:** Al transformar el *log-odds* en probabilidad mediante la función logística $\sigma(z) = \frac{1}{1 + e^{-z}}$, el factor multiplicativo $m$ empuja el argumento $z$ hacia $+\infty$ o $-\infty$, colapsando $\hat{P}(Y=1 \mid \mathbf{x}) \to 1$ o $0$.

### ¿Cuándo es Crítica la Calibración en la Industria?
Un clasificador está **bien calibrado** si, de todas las muestras a las que les asigna una probabilidad $\hat{p} = 0.80$, aproximadamente el **80% pertenecen realmente a la clase positiva**:

$$P\left(Y = 1 \mid \hat{P}(Y=1 \mid \mathbf{X}) = p\right) = p \quad \forall p \in [0, 1]$$

Esto es indispensable en:
* **Riesgo Crediticio y Seguros:** Cálculo de Pérdida Esperada ($\text{EL} = \text{PD} \times \text{LGD} \times \text{EAD}$).
* **Publicidad Digital (*AdTech*):** Subastas en tiempo real (*Real-Time Bidding*) donde la puja es $\text{Bid} = P(\text{Click} \mid \mathbf{x}) \times \text{Valor}_{\text{Conversión}}$.
* **Diagnóstico Médico:** Comunicación de certidumbre clínica al especialista.

In [ ]:
# Configuración del entorno y librerías estándar
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import StandardScaler
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    brier_score_loss, log_loss
)

# Estándar visual del repositorio
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

print("Entorno configurado correctamente.")

---

## 2. Evaluación y Corrección de la Calibración: Curvas de Confiabilidad, Brier Score y `CalibratedClassifierCV`

### Métrica Cuantitativa: El Puntaje de Brier (*Brier Score*)
El **Brier Score** mide el Error Cuadrático Medio (*MSE*) entre las probabilidades predichas $\hat{p}_i = \hat{P}(Y=1 \mid \mathbf{x}_i)$ y las etiquetas binarias reales $y_i \in \{0, 1\}$:

$$\text{BS} = \frac{1}{N} \sum_{i=1}^{N} \left(y_i - \hat{p}_i\right)^2$$

* Es una **regla de puntuación estrictamente propia** (*strictly proper scoring rule*): descompone simultáneamente la **calibración** (confiabilidad) y el **refinamiento** (resolución/discriminación).
* **Rango:** $[0, 1]$, donde **`0.0` es perfecto** y un modelo que predice siempre $0.5$ obtiene $\text{BS} = 0.25$.

### Métodos de Recalibración Post-Hoc en `CalibratedClassifierCV`
Dado el puntaje crudo no calibrado $s_i = \hat{P}_{\text{NB}}(Y=1 \mid \mathbf{x}_i)$ devuelto por Naive Bayes sobre un conjunto de validación disjunto (o vía *Cross-Validation* interna para evitar *data leakage*), ajustamos una función monótona $g: [0, 1] \to [0, 1]$:

1. **Escalado de Platt (`method='sigmoid'`):**  
   Ajusta una regresión logística univariada de dos parámetros $(A, B \in \mathbb{R})$ sobre las salidas del clasificador:
   $$P(Y = 1 \mid s_i) = \frac{1}{1 + \exp(A \cdot s_i + B)}$$
   * **Cuándo usarlo:** Ideal cuando la distorsión en la curva de confiabilidad tiene forma de **"S" sigmoidal** y el conjunto de datos de calibración es pequeño o mediano ($N < 1,000$ muestras), ya que al tener solo 2 parámetros no sobreajusta.
2. **Regresión Isotónica (`method='isotonic'`):**  
   Ajusta una función escalonada constante a trozos no decreciente ($\hat{m}(s_i) \le \hat{m}(s_j) \; \forall s_i < s_j$) mediante el algoritmo *Pool Adjacent Violators Algorithm* (PAVA):
   $$\underset{m_1 \le m_2 \le \dots \le m_N}{\min} \sum_{i=1}^{N} \left(y_i - m(s_i)\right)^2$$
   * **Cuándo usarlo:** Método no paramétrico capaz de corregir cualquier distorsión monótona arbitraria. Requiere **datasets medianos o grandes ($N > 1,000$ muestras)** para evitar sobreajustar los escalones.

In [ ]:
# Experimento de Calibración: GaussianNB Crudo vs Platt Scaling (Sigmoid) vs Regresión Isotónica
np.random.seed(42)

# Generamos un dataset con alta redundancia (n_redundant=12) que induce fuerte colinealidad
X_cal, y_cal = make_classification(
    n_samples=6000, n_features=20, n_informative=6, n_redundant=12,
    n_clusters_per_class=2, weights=[0.5, 0.5], random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cal, y_cal, test_size=0.30, stratify=y_cal, random_state=42
)

# Modelos a comparar
gnb_uncal = GaussianNB().fit(X_train_c, y_train_c)
gnb_sigmoid = CalibratedClassifierCV(GaussianNB(), method='sigmoid', cv=5).fit(X_train_c, y_train_c)
gnb_isotonic = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=5).fit(X_train_c, y_train_c)
lr_ref = LogisticRegression(max_iter=1000).fit(X_train_c, y_train_c)

modelos_cal = [
    ("GaussianNB (Sin Calibrar)", gnb_uncal, '#d62728', '-'),
    ("GaussianNB + Sigmoid (Platt)", gnb_sigmoid, '#ff7f0e', '--'),
    ("GaussianNB + Isotonic", gnb_isotonic, '#2ca02c', '-.'),
    ("Regresión Logística (Referencia)", lr_ref, '#1f77b4', ':')
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

tabla_metricas_cal = []

for nombre, clf, color, ls in modelos_cal:
    y_prob = clf.predict_proba(X_test_c)[:, 1]
    y_pred = clf.predict(X_test_c)
    
    brier = brier_score_loss(y_test_c, y_prob)
    lloss = log_loss(y_test_c, y_prob)
    acc = accuracy_score(y_test_c, y_pred)
    auc = roc_auc_score(y_test_c, y_prob)
    
    tabla_metricas_cal.append({
        'Modelo': nombre,
        'Brier Score (↓ mejor)': brier,
        'Log-Loss (↓ mejor)': lloss,
        'Accuracy (↑ mejor)': acc,
        'ROC-AUC (↑ mejor)': auc
    })
    
    # Curva de Confiabilidad (Reliability Diagram)
    CalibrationDisplay.from_estimator(
        clf, X_test_c, y_test_c, n_bins=10, name=f"{nombre} (BS={brier:.3f})",
        ax=axes[0], color=color, linestyle=ls
    )
    
    # Histograma de probabilidades predichas
    axes[1].hist(y_prob, bins=25, range=(0, 1), histtype='step', lw=2.3,
                 label=nombre, color=color, linestyle=ls, density=True)

axes[0].set_title("Curvas de Confiabilidad (Reliability Diagrams)", fontsize=12.5, fontweight='bold')
axes[0].set_xlabel("Probabilidad Media Predicha", fontsize=11)
axes[0].set_ylabel("Fracción Real de Positivos", fontsize=11)
axes[0].legend(loc='upper left', frameon=True, facecolor='white', fontsize=9.5)

axes[1].set_title("Distribución de Probabilidades Predichas $\hat{P}(Y=1 \mid X)$", fontsize=12.5, fontweight='bold')
axes[1].set_xlabel("Probabilidad Predicha", fontsize=11)
axes[1].set_ylabel("Densidad", fontsize=11)
axes[1].legend(loc='upper center', frameon=True, facecolor='white', fontsize=9.5)

plt.tight_layout()
plt.show()

df_cal = pd.DataFrame(tabla_metricas_cal).set_index('Modelo')
print("=== AUDITORÍA CUANTITATIVA DE CALIBRACIÓN EN TEST ===")
display(df_cal.style.format({
    'Brier Score (↓ mejor)': '{:.4f}',
    'Log-Loss (↓ mejor)': '{:.4f}',
    'Accuracy (↑ mejor)': '{:.2%}',
    'ROC-AUC (↑ mejor)': '{:.4f}'
}))

---

## 3. Aprendizaje Fuera de Núcleo (*Out-of-Core / Streaming*) con `partial_fit` y `HashingVectorizer`

### El Reto del Big Data en una Sola Máquina
¿Qué sucede cuando necesitamos entrenar un clasificador sobre un archivo de logs o transacciones de **100 GB** (o un flujo continuo de Apache Kafka) y nuestra estación de trabajo solo dispone de **8 GB o 16 GB de RAM**?
* Algoritmos como `RandomForestClassifier`, `SVC` o `KNeighborsClassifier` requieren cargar la matriz de diseño completa en memoria RAM.
* En cambio, **todas las variantes de Naive Bayes en Scikit-Learn (`GaussianNB`, `MultinomialNB`, `BernoulliNB`, `ComplementNB`, `CategoricalNB`) implementan el método `.partial_fit(X_batch, y_batch, classes=...)`**.

### Por Qué Naive Bayes se Actualiza Exactamente en $\mathcal{O}(1)$ de Memoria:
1. **En modelos discretos (`MultinomialNB`):** Las estadísticas suficientes son únicamente los conteos aditivos $N_{kj}$ y $N_k$. Cuando llega un nuevo lote (*mini-batch* o *chunk*) $B$, basta con sumar los conteos del lote a los acumuladores existentes:
   $$N_{kj}^{(t+1)} = N_{kj}^{(t)} + \sum_{i \in B_k} x_{ij}$$
2. **En `GaussianNB`:** Actualiza las medias $\mu_{kj}$ y varianzas $\sigma_{kj}^2$ por lotes utilizando las fórmulas incrementalmente estables de **Chan, Golub & LeVeque (generalización por lotes del algoritmo de Welford)**, sin almacenar jamás las muestras pasadas.

### El Compañero Indispensable en NLP Streaming: `HashingVectorizer`
En un flujo de texto continuo, `CountVectorizer` no sirve para *streaming* puro porque requiere una primera pasada completa sobre todo el dataset para construir el diccionario de vocabulario en memoria (y el vocabulario crece sin límite con el tiempo).  
**`HashingVectorizer`** aplica una función hash criptográfica/rápida (MurmurHash3) directamente sobre cada cadena de texto para proyectarla en una matriz dispersa de dimensión fija $D = 2^b$ (`n_features`), logrando **cero estado en memoria** (*stateless*).

In [ ]:
# Simulación de Arquitectura Streaming Out-of-Core: Generador de Chunks + HashingVectorizer + partial_fit
np.random.seed(42)

vocab_stream = {
    0: ['normal', 'login', 'consulta', 'saldo', 'transferencia', 'recibo', 'pago', 'autorizado', 'ok', 'cliente'],
    1: ['alerta', 'exploit', 'sql_injection', 'payload', 'root', 'fuerza_bruta', 'timeout', 'bloqueo', 'malware', 'ddos']
}
vocab_ruido_stream = ['ip', 'puerto', 'servidor', 'request', 'response', 'http', 'sesion', 'token', 'timestamp']

def stream_logs_generator(n_batches: int, batch_size: int = 400, seed: int = 42):
    """Generador perezoso (lazy generator) que simula lectura por lotes desde disco o Kafka sin usar RAM."""
    rng = np.random.default_rng(seed)
    for b in range(n_batches):
        textos_batch, y_batch = [], []
        for _ in range(batch_size):
            clase = rng.choice([0, 1], p=[0.80, 0.20]) # 20% eventos de ataque
            n_palabras = rng.integers(8, 18)
            # 60% señal de la clase, 40% ruido común (con 10% de contaminación cruzada)
            clase_efectiva = clase if rng.random() > 0.12 else (1 - clase)
            t_senal = rng.choice(vocab_stream[clase_efectiva], size=int(n_palabras*0.6), replace=True)
            t_ruido = rng.choice(vocab_ruido_stream, size=n_palabras - len(t_senal), replace=True)
            log_line = " ".join(np.concatenate([t_senal, t_ruido]))
            textos_batch.append(log_line)
            y_batch.append(clase)
        yield b + 1, textos_batch, np.array(y_batch)

# Vectorizador sin estado en memoria (Stateless) con 2^12 = 4096 columnas y no negatividad para MultinomialNB
hasher = HashingVectorizer(n_features=2**12, alternate_sign=False, norm=None)

# Reservamos un conjunto de validación fijo para medir la evolución del aprendizaje
_, textos_val_stream, y_val_stream = next(stream_logs_generator(1, batch_size=1200, seed=999))
X_val_stream = hasher.transform(textos_val_stream)

# Instanciamos MultinomialNB para entrenamiento incremental
clf_streaming = MultinomialNB(alpha=0.5)
clases_totales = np.array([0, 1])

historial_batches = []
muestras_acumuladas = 0

for batch_id, textos_b, y_b in stream_logs_generator(n_batches=25, batch_size=300, seed=42):
    t0 = time.perf_counter()
    X_b = hasher.transform(textos_b)
    # IMPORTANTE: En el primer lote es obligatorio pasar el parámetro classes=clases_totales
    clf_streaming.partial_fit(X_b, y_b, classes=clases_totales)
    t_batch_ms = (time.perf_counter() - t0) * 1000
    
    muestras_acumuladas += len(y_b)
    y_pred_val = clf_streaming.predict(X_val_stream)
    y_prob_val = clf_streaming.predict_proba(X_val_stream)[:, 1]
    
    historial_batches.append({
        'Batch': batch_id,
        'Muestras_Acumuladas': muestras_acumuladas,
        'F1_Score_Ataques': f1_score(y_val_stream, y_pred_val),
        'ROC_AUC': roc_auc_score(y_val_stream, y_prob_val),
        'Tiempo_Batch_ms': t_batch_ms
    })

df_stream = pd.DataFrame(historial_batches)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].plot(df_stream['Muestras_Acumuladas'], df_stream['F1_Score_Ataques'], marker='o', color='#1f77b4', lw=2.2, label='F1-Score (Detección Ataques)')
axes[0].plot(df_stream['Muestras_Acumuladas'], df_stream['ROC_AUC'], marker='s', color='#2ca02c', lw=2.2, label='ROC-AUC en Validación')
axes[0].set_title('Convergencia Incremental Out-of-Core con partial_fit', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Muestras Procesadas en Streaming', fontsize=11)
axes[0].set_ylabel('Métrica en Conjunto de Validación', fontsize=11)
axes[0].legend(frameon=True, facecolor='white')

axes[1].bar(df_stream['Batch'], df_stream['Tiempo_Batch_ms'], color='#ff7f0e', alpha=0.85, edgecolor='k')
axes[1].axhline(df_stream['Tiempo_Batch_ms'].mean(), color='#d62728', linestyle='--', lw=2, label=f"Promedio: {df_stream['Tiempo_Batch_ms'].mean():.2f} ms/lote")
axes[1].set_title('Latencia de Actualización por Lote (Complejidad O(1) por Muestra)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Número de Lote (Batch de 300 logs)', fontsize=11)
axes[1].set_ylabel('Tiempo de Vectorización + partial_fit (ms)', fontsize=11)
axes[1].legend(frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

---

## 4. Benchmark Exhaustivo de Clasificación: Calidad, Tiempo de Entrenamiento y Latencia de Inferencia

Para tomar decisiones de arquitectura fundamentadas en producción, enfrentaremos a **Naive Bayes** (tanto crudo como calibrado) contra las principales familias algorítmicas estudiadas en el repositorio:
1. **Regresión Logística (`LogisticRegression`):** Clasificador lineal discriminativo optimizado vía L-BFGS.
2. **K-Vecinos Más Cercanos (`KNeighborsClassifier`):** Aprendizaje basado en instancias (perezoso / *lazy learner*).
3. **Árbol de Decisión (`DecisionTreeClassifier` - CART):** Particionamiento recursivo ortogonal.
4. **Bosque Aleatorio (`RandomForestClassifier`):** Ensamble Bagging de árboles profundos.
5. **Gaussian Naive Bayes (`GaussianNB`)** y **GaussianNB Calibrado (`CalibratedClassifierCV`)**.

Evaluaremos tres dimensiones críticas:
* **Calidad Predictiva y Probabilística:** Exactitud (*Accuracy*), *F1-Weighted*, *ROC-AUC* y *Brier Score*.
* **Escalabilidad de Entrenamiento:** Curva de tiempo de `.fit()` al crecer el tamaño muestral $N$ desde $1,000$ hasta $50,000$ registros.
* **Latencia de Inferencia en Tiempo Real:** Microsegundos ($\mu\text{s}$) requeridos por cada muestra durante `.predict_proba()`.

In [ ]:
# 1. Benchmark de Calidad Predictiva, Tiempo de Entrenamiento y Latencia de Inferencia (N = 15,000, D = 25)
np.random.seed(42)

X_bench, y_bench = make_classification(
    n_samples=15000, n_features=25, n_informative=15, n_redundant=5,
    n_classes=2, class_sep=1.15, flip_y=0.03, random_state=42
)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bench, y_bench, test_size=0.25, stratify=y_bench, random_state=42
)

# Estandarizamos para que KNN y Regresión Logística compitan en igualdad de condiciones
scaler_b = StandardScaler()
X_tr_b = scaler_b.fit_transform(X_tr_b)
X_te_b = scaler_b.transform(X_te_b)

catalogo_modelos = {
    'GaussianNB (Crudo)': GaussianNB(),
    'GaussianNB (Calibrado Platt)': CalibratedClassifierCV(GaussianNB(), method='sigmoid', cv=3),
    'Regresión Logística': LogisticRegression(max_iter=500, random_state=42),
    'Árbol CART (max_depth=10)': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN (k=15)': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    'Random Forest (100 árboles)': RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
}

resultados_benchmark = []
n_test_samples = X_te_b.shape[0]

for nombre, modelo in catalogo_modelos.items():
    # Medición de tiempo de entrenamiento
    t0_fit = time.perf_counter()
    modelo.fit(X_tr_b, y_tr_b)
    t_fit_ms = (time.perf_counter() - t0_fit) * 1000
    
    # Medición de latencia de inferencia probabilística
    t0_pred = time.perf_counter()
    y_prob = modelo.predict_proba(X_te_b)[:, 1]
    t_pred_total_s = time.perf_counter() - t0_pred
    latencia_us_por_muestra = (t_pred_total_s / n_test_samples) * 1e6
    
    y_pred = (y_prob >= 0.5).astype(int)
    
    resultados_benchmark.append({
        'Familia / Modelo': nombre,
        'Accuracy': accuracy_score(y_te_b, y_pred),
        'F1-Weighted': f1_score(y_te_b, y_pred, average='weighted'),
        'ROC-AUC': roc_auc_score(y_te_b, y_prob),
        'Brier Score': brier_score_loss(y_te_b, y_prob),
        'Tiempo Train (ms)': t_fit_ms,
        'Latencia Inferencia (µs/muestra)': latencia_us_por_muestra
    })

df_bench = pd.DataFrame(resultados_benchmark).set_index('Familia / Modelo')
print("=== TABLA MAESTRA DE BENCHMARK (N_train = 11,250 | N_test = 3,750 | D = 25) ===")
display(df_bench.style.format({
    'Accuracy': '{:.2%}',
    'F1-Weighted': '{:.4f}',
    'ROC-AUC': '{:.4f}',
    'Brier Score': '{:.4f}',
    'Tiempo Train (ms)': '{:.2f} ms',
    'Latencia Inferencia (µs/muestra)': '{:.2f} µs'
}))

In [ ]:
# 2. Curvas de Escalabilidad Asintótica en Entrenamiento (Variando N desde 1,000 hasta 40,000)
tamanos_N = [1000, 3000, 7500, 15000, 25000, 40000]
modelos_escalabilidad = {
    'GaussianNB O(N·D)': GaussianNB(),
    'Regresión Logística': LogisticRegression(max_iter=300),
    'Árbol CART': DecisionTreeClassifier(max_depth=10),
    'Random Forest (50 árb.)': RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=1)
}

tiempos_por_N = {k: [] for k in modelos_escalabilidad}

X_large, y_large = make_classification(n_samples=max(tamanos_N), n_features=25, random_state=42)

for N_sub in tamanos_N:
    X_sub, y_sub = X_large[:N_sub], y_large[:N_sub]
    for nombre_m, clf_m in modelos_escalabilidad.items():
        t_ini = time.perf_counter()
        clf_m.fit(X_sub, y_sub)
        tiempos_por_N[nombre_m].append((time.perf_counter() - t_ini) * 1000)

# Visualización Comparativa: Escalabilidad vs Trade-off Latencia/ROC-AUC
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
colores_esc = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

# Panel 1: Escalabilidad con N
for (nombre_m, lista_t), col in zip(tiempos_por_N.items(), colores_esc):
    axes[0].plot(tamanos_N, lista_t, marker='o', lw=2.5, label=nombre_m, color=col)

axes[0].set_yscale('log')
axes[0].set_title('Tiempo de Entrenamiento vs Tamaño Muestral $N$ (Escala Log)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Número de Muestras de Entrenamiento ($N$)', fontsize=11)
axes[0].set_ylabel('Tiempo de Entrenamiento en ms (Escala Log)', fontsize=11)
axes[0].legend(frameon=True, facecolor='white', fontsize=10)

# Panel 2: Frontera de Pareto (Latencia de Inferencia vs ROC-AUC)
for idx, row in df_bench.iterrows():
    axes[1].scatter(row['Latencia Inferencia (µs/muestra)'], row['ROC-AUC'], s=130, edgecolor='k', zorder=5)
    axes[1].annotate(
        idx, (row['Latencia Inferencia (µs/muestra)'], row['ROC-AUC']),
        xytext=(8, -4), textcoords='offset points', fontsize=9.5, fontweight='bold'
    )

axes[1].set_xscale('log')
axes[1].set_title('Trade-off en Producción: Latencia de Inferencia vs. ROC-AUC', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Latencia de Inferencia por Muestra en µs (Escala Log - Menor es más rápido)', fontsize=11)
axes[1].set_ylabel('Área Bajo la Curva ROC (ROC-AUC)', fontsize=11)
axes[1].set_xlim(left=df_bench['Latencia Inferencia (µs/muestra)'].min() * 0.5,
                 right=df_bench['Latencia Inferencia (µs/muestra)'].max() * 8.0)

plt.tight_layout()
plt.show()

---

## 5. Cuadro Comparativo Final y Guía de Decisión Arquitectural para Producción

| Criterio Arquitectónico | Naive Bayes (`GaussianNB` / `MultinomialNB`) | Regresión Logística | K-Nearest Neighbors (`KNN`) | Árboles CART (`DecisionTree`) | Random Forest (`RandomForest`) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Paradigma** | Generativo $P(\mathbf{X}, Y)$ | Discriminativo $P(Y \mid \mathbf{X})$ | Basado en Instancias (*Lazy*) | Discriminativo (Reglas Ortogonales) | Discriminativo (Ensamble *Bagging*) |
| **Complejidad de Entrenamiento** | **$\mathcal{O}(N \cdot D)$ (1 sola pasada)** | $\mathcal{O}(N \cdot D \cdot n_{\text{iter}})$ | $\mathcal{O}(1)$ o $\mathcal{O}(N \cdot D \log N)$ (KD-Tree) | $\mathcal{O}(N \cdot D \log N)$ | $\mathcal{O}(T \cdot N \cdot \sqrt{D} \log N)$ |
| **Complejidad de Inferencia** | **$\mathcal{O}(C \cdot D)$ (Ultra rápida)** | $\mathcal{O}(C \cdot D)$ (Ultra rápida) | $\mathcal{O}(N \cdot D)$ (**Muy lenta en prod**) | $\mathcal{O}(\text{profundidad})$ (Rápida) | $\mathcal{O}(T \cdot \text{profundidad})$ (Moderada) |
| **Aprendizaje Streaming (`partial_fit`)** | **Sí (Nativo y Exacto)** | Sí (Vía `SGDClassifier`) | No | No | No |
| **Calibración Nativa de Probabilidades** | **Pobre (Sobreconfiado $\to 0/1$)** | **Excelente (Minimiza Log-Loss)** | Moderada | Pobre en árboles profundos | Moderada (Requiere calibración en colas) |
| **Tolerancia a Colinealidad** | Se degrada en calibración | Requiere regularización L2 | Sensible a características redundantes | Robusto | Muy robusto |
| **Escalado de Variables (`StandardScaler`)** | Invariante (salvo `var_smoothing` grande) | **Obligatorio** (con regularización) | **Obligatorio** | Invariante | Invariante |

### Reglas de Oro para el Arquitecto de Machine Learning:
1. **¿Cuándo elegir Naive Bayes como modelo principal en producción?**
   * Sistemas de **baja latencia extrema** (< 5 microsegundos por predicción) o clasificación sobre matrices dispersas de muy alta dimensionalidad (NLP, filtrado de logs, clasificación documental).
   * Escenarios con **pocos datos etiquetados ($N$ pequeño)** donde los modelos discriminativos sobreajustan rápidamente.
   * Flujos de datos masivos o continuos (*Streaming* / Kafka) donde el modelo debe actualizarse en línea con `.partial_fit()`.
2. **¿Cuándo evitar Naive Bayes o envolverlo obligatoriamente?**
   * Si el problema depende de **interacciones no lineales complejas entre variables** (ej. problemas tipo XOR donde ninguna variable por separado discrimina la clase, sino su combinación conjunta), Naive Bayes fallará y debes preferir **Random Forest / Gradient Boosting**.
   * Si las probabilidades posteriores se usarán para decisiones financieras o cálculo de valor esperado, **envuelve siempre a Naive Bayes dentro de `CalibratedClassifierCV`**.

---
**Siguiente paso:** En el **Notebook 05 (`05_Ejercicios_Naive_Bayes.ipynb`)** pondrás a prueba todos los conceptos matemáticos y de ingeniería aprendidos mediante 7 desafíos prácticos progresivos.